# Playground: talk to `base_100m`

This is a **base model**: it continues text, it doesn't follow instructions. Start the text you want it to write:
`Photosynthesis is the process by which` works much better than `What is photosynthesis?`.

Runs fine on a **CPU runtime** (Runtime → Change runtime type → CPU), which costs almost no compute units.
Needs the `HF_TOKEN` secret (for the tokenizer) and the checkpoint from the training notebook on Drive.

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
CHECKPOINT = "/content/drive/MyDrive/transformer-from-scratch/base_100m/best.pt"

import os
REPO = "/content/transformer-pytorch"
if not os.path.exists(REPO):
    !git clone -q https://github.com/efe-tatar/transformer-pytorch.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q regex

In [ ]:
import sys, torch
from huggingface_hub import hf_hub_download
from config import ModelConfig
from network.causal_lm import CausalLM
from tokenizer.bpe import BPETokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
model = CausalLM(ModelConfig(**ckpt["model_cfg"]))
model.load_state_dict(ckpt["model"])
model.to(device).eval()

tokenizer = BPETokenizer(0)
tokenizer.load(hf_hub_download("Efe-Tatar/fineweb-edu-3b-bpe32k", "tokenizer.json", repo_type="dataset",
                               token=userdata.get("HF_TOKEN")))
EOT = tokenizer.special_tokens["<|endoftext|>"]
print(f"loaded step {ckpt['step']} on {device} | val loss {ckpt.get('best_val', float('nan')):.3f}")

In [ ]:
@torch.no_grad()
def stream(prompt, max_new_tokens=200, temperature=0.8, top_k=50):
    # same sampling as CausalLM.generate, but prints each token as it arrives
    ids = torch.tensor([[EOT] + tokenizer.encode(prompt)], device=device)
    max_new_tokens = min(max_new_tokens, model.max_seq_len - ids.shape[1])
    logits, cache, n = model(ids, use_cache=True)
    out, printed = [], ""
    for _ in range(max_new_tokens):
        next_logits = logits[:, -1, :].float() / temperature
        kth = torch.topk(next_logits, top_k).values[:, -1:]
        next_logits = next_logits.masked_fill(next_logits < kth, float("-inf"))
        token = torch.multinomial(torch.softmax(next_logits, dim=-1), 1)
        if token.item() == EOT:
            break
        out.append(token.item())
        text = tokenizer.decode(out)
        # a multi-byte character can be split across tokens: wait until it's complete
        if not text.endswith("\ufffd"):
            print(text[len(printed):], end="", flush=True)
            printed = text
        logits, cache, n = model(token, use_cache=True, kv_cache_list=cache, cached_seq_len=n)
    print()

stream("Photosynthesis is the process by which")

## Chat loop

Type the beginning of a text, the model continues it. Empty line to stop.
Lower `temperature` (e.g. 0.5) = safer and more repetitive, higher (1.0) = more creative and more chaotic.

In [ ]:
TEMPERATURE = 0.8
while True:
    prompt = input("\nyou> ")
    if not prompt.strip():
        break
    print("model> " + prompt, end="")
    stream(prompt, temperature=TEMPERATURE)